# Implementing a FNO for BIP in python

Will try to follow https://github.com/neuraloperator/neuraloperator for examples.

Specifically see their "examples" page.

In [ ]:
# Imports

import torch
import matplotlib.pyplot as plt
import numpy as np
import sys
import neuralop
from neuralop.models import FNO
from neuralop import Trainer
from neuralop.training import AdamW
from neuralop.data.datasets import load_darcy_flow_small, DarcyDataset
from neuralop.utils import count_model_params
from neuralop import LpLoss, H1Loss
from torch.utils.data import DataLoader

In [ ]:
# GPU activation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Loading the Darcy-Flow dataset
n_train = 1000
n_tests = [100, 50]
train_resolution = 16
test_resolutions = [16, 32]
batch_size = 32
test_batch_sizes = [32, 32]

dataset = DarcyDataset(
    root_dir="./darcy_data",
    n_train=n_train,
    n_tests=n_tests,
    train_resolution=train_resolution,
    batch_size=batch_size,
    test_resolutions=test_resolutions,
    test_batch_sizes=test_batch_sizes,
)

train_loader = DataLoader(
        dataset.train_db,
        batch_size=batch_size,
        num_workers=1,
        pin_memory=True,
        persistent_workers=False,
    )

test_loaders = {}
for res, test_bsize in zip(test_resolutions, test_batch_sizes):
    test_loaders[res] = DataLoader(
        dataset.test_dbs[res],
        batch_size=test_bsize,
        shuffle=False,
        num_workers=1,
        pin_memory=True,
        persistent_workers=False,
    )

data_processor = dataset.data_processor

In [ ]:
# Visualize a sample from the training set
i = 0
print(type(train_loader), train_loader.__len__())
for sample in train_loader:
    if i == 10:
        test_sample = sample
        break
    i += 1

x = test_sample['x']  # shape (32, 1, 16, 16) - input θ
y = test_sample['y']  # shape (32, 1, 16, 16) - solution u

print(x.shape, y.shape)
# pick the first sample
theta = x[0, 0].numpy()  # shape (16, 16)
u     = y[0, 0].numpy()  # shape (16, 16)

print(theta)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(theta, cmap='viridis')
axes[0].set_title("Input: θ (coefficient field)")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(u, cmap='plasma')
axes[1].set_title("Output: u (solution field)")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# Creating the FNO model
# ----------------------

model = FNO(
    n_modes=(16, 16),
    in_channels=1,
    out_channels=1,
    hidden_channels=32,
    projection_channel_ratio=2,
    factorization="tucker",
    rank=0.42,
)

# NOTE: FNO takes in inputs of shape (batch_size, in_channels, height, width) and outputs the same shape.
# If you want to pass a single observation, you need to add batch and channel dimensions, e.g., x[0, 0].unsqueeze(0).unsqueeze(0) to get shape (1, 1, 16, 16).

model = model.to(device)

n_params = count_model_params(model)
print(f"\nOur model has {n_params} parameters.")
sys.stdout.flush()


# Creating the optimizer and scheduler
# ------------------------------------
optimizer = AdamW(model.parameters(), lr=8e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)


# Creating the losses
# ------------------
l2loss = LpLoss(d=2, p=2)
h1loss = H1Loss(d=2)

train_loss = h1loss
eval_losses = {"h1": h1loss, "l2": l2loss}


In [ ]:
# Trainer hyperparameters
epochs = 100

In [ ]:
# Creating the trainer
# --------------------
trainer = Trainer(
    model=model,
    n_epochs=epochs,
    device=device,
    data_processor=data_processor,
    wandb_log=False,
    eval_interval=3,
    use_distributed=False,
    verbose=True,
)

# We train and save checkpoints

trainer.train(
    train_loader=train_loader,
    test_loaders={},
    optimizer=optimizer,
    scheduler=scheduler,
    regularizer=False,
    training_loss=train_loss,
    save_every=1,
    save_dir="./checkpoints",
)

In [ ]:
torch.serialization.add_safe_globals([torch._C._nn.gelu])
torch.serialization.add_safe_globals([neuralop.layers.spectral_convolution.SpectralConv])
# .. resume_from_dir:
# resume training from saved checkpoint at epoch 10

trainer = Trainer(
    model=model,
    n_epochs=epochs,
    device=device,
    data_processor=data_processor,
    wandb_log=False,
    eval_interval=3,
    use_distributed=False,
    verbose=True,
)

trainer.train(
    train_loader=train_loader,
    test_loaders={},
    optimizer=optimizer,
    scheduler=scheduler,
    regularizer=False,
    training_loss=train_loss,
    resume_from_dir="./checkpoints",
)

In [ ]:
# visualise output of FNO after training on our test_sample
with torch.no_grad():
    u_pred = model(test_sample['x'].to(device))

u_pred_np = u_pred[0, 0].cpu().numpy()  # need .cpu() before .numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(theta, cmap='viridis')
axes[0].set_title("Input: θ (coefficient field)")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(u_pred_np, cmap='plasma')
axes[1].set_title("Output: post-trained predicted u (solution field)")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()


In [ ]:
sample16 = theta
output16 = u_pred_np

In [ ]:
# super-resolution: test on 32x32 to observe function truncation effects


# Visualize a sample from the training set
for sample in test_loaders[32]:
    test_sample32 = sample
    break

# super-resolution output
with torch.no_grad():
    u_pred32 = model(test_sample32['x'].to(device))

u_pred32_np = u_pred32[0, 0].cpu().numpy()  # need .cpu() before .numpy()

x32 = test_sample32['x']  # shape (32, 1, 16, 16) - input θ
y32 = test_sample32['y']  # shape (32, 1, 16, 16) - solution u

# pick the first sample
theta32 = x32[0, 0].numpy()  # shape (16, 16)
u32     = y32[0, 0].numpy()  # shape (16, 16)

print(theta32)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].imshow(theta32, cmap='viridis')
axes[0].set_title("Input: θ (coefficient field)")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(u32, cmap='plasma')
axes[1].set_title("Output: true u (solution field)")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(u_pred32_np, cmap='plasma')
axes[2].set_title("Output: super-resolution u (solution field)")
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
# Parameters for MCMC
delta = 0.0025
n_iterations = 10_000  # this is essentially a burn in period
accepted = 0  # counter for accepted proposals

In [ ]:
# Now we have our model, do the inverse problem with the same dataset.

def log_likelihood(x: torch.Tensor, y: torch.Tensor, model) -> torch.Tensor:
    """Inputs:
    - x: input tensor (location measurements of field θ)
    - y: observed output tensor (e.g., solution field u at locations x)
    - model: the trained FNO model that maps x to predicted y
    Returns:
    - log-likelihood of observing y given x under the model's predictions"""
    # Ensure x and y are 4D tensors with shape (1, 1, height, width)
    # Ensure both x and y are on the same device as the model
    x = x.to(device)
    y = y.to(device)
    # Assuming Gaussian likelihood with fixed variance
    with torch.no_grad():
        forwards = model(x)
    forwards = forwards[0, 0]  # need .cpu() before .numpy()
    residuals = (y - forwards).cpu().numpy()
    ll = -0.5 * np.sum(residuals**2)
    return ll


In [ ]:
true_field = torch.tensor(sample16, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # shape (1, 1, 16, 16)
observation = torch.tensor(output16, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # shape (1, 1, 16, 16)
print(true_field.shape, observation.shape)
print(log_likelihood(true_field, observation, model))

In [ ]:
# Recall that the input field is consisting of 0 and 1
# Can eventually try to implement more sophisticated MCMC algorithm based on A. Stuart's "Geometric MCMC for Infinite-Dimensional Inverse Problems"
# Technically the random variable xi~N(0, C) is the log-GP as we assume in darcy flow 

# Sample from GP
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern

def sample_GP(dimension=16,kernel=RBF, display_field=False):
    # create 2D grid over [0,1] x [0,1]
    x_coords = np.linspace(0, 1, dimension)
    y_coords = np.linspace(0, 1, dimension)
    X, Y = np.meshgrid(x_coords, y_coords)

    # flatten grid to (1024, 2) for GP input
    grid_points = np.column_stack([X.ravel(), Y.ravel()])

    kernel = RBF(length_scale=0.3)
    gp = GaussianProcessRegressor(kernel=kernel)

    # sample one realisation
    sample = gp.sample_y(grid_points, n_samples=1, random_state=None)  # pass keyword random_state=None to obtain other sdifferent sample each time, otherwise it will be the same sample each time
    sample_2d = sample.reshape(dimension, dimension)

    if display_field:
        plt.imshow(sample_2d, cmap='viridis')
        plt.colorbar()
        plt.title("GP sample over [0,1]²")
        plt.show()

    return sample_2d


def smooth_threshold(x, tau=10, display_field=False):
    """A smooth approximation to the step function, which can be used to get a binary field from the GP sample while maintaining differentiability.
    
    See Akiyldiz for details.
    """
    normalising = np.linalg.norm(x)
    thresholded = 0.5 * np.tanh(tau * x) + 0.5
    if display_field:
        plt.imshow(thresholded, cmap='viridis')
        plt.colorbar()
        plt.title("Smooth thresholded field")
        plt.show()
    return thresholded


In [ ]:
u_0 = sample_GP(display_field=True)  # Initial proposal
theta_0 = smooth_threshold(u_0, display_field=True)  # threshold to get binary field
print(u_0.shape)

theta_evolution = [theta_0]  # to store the evolution of theta
loss_evolution = []
accepted = 0
for i in range(n_iterations):
    # single MCMC step
    u_proposal = np.sqrt(1-2*delta)*u_0 + np.sqrt(2*delta)*sample_GP()
    theta_proposal = smooth_threshold(u_proposal)  # threshold to get binary field

    # Make proposals pytorch tensors, and add batch and channel dimensions to match FNO input shape (1, 1, 16, 16)
    theta_proposal_tensor = torch.tensor(theta_proposal, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    theta_0_tensor = torch.tensor(theta_0, dtype=torch.float32).unsqueeze(0).unsqueeze(0)


    acceptance_prob = min(1, np.exp(log_likelihood(theta_proposal_tensor, observation, model) - log_likelihood(theta_0_tensor, observation, model)))
    if np.random.rand() < acceptance_prob:
        theta_0 = theta_proposal
        u_0 = u_proposal
        accepted += 1
        theta_evolution.append(theta_0)
    theta_evolution.append(theta_0)
    loss_evolution.append(log_likelihood(theta_0_tensor, observation, model).item())
    if (i+1) % 100 == 0:
        print(f"Iteration {i+1}/{n_iterations}, Acceptance Rate: {accepted/(i+1):.3f}")



In [ ]:
print(f"final loss: {log_likelihood(torch.tensor(theta_evolution[-1], dtype=torch.float32).unsqueeze(0).unsqueeze(0), observation, model)}")
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

im0 = axes[0].imshow(theta_evolution[-1], cmap='viridis')
axes[0].set_title("Final computed field")
plt.colorbar(im0, ax=axes[0])

with torch.no_grad():
    theta_pred = model(torch.tensor(theta_evolution[-1], dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device))

im1 = axes[1].imshow(theta_pred[0,0].cpu().numpy(), cmap='plasma')
axes[1].set_title("output field from predicted theta")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# plot loss
plt.plot(loss_evolution)
plt.title("Log-likelihood evolution")
plt.xlabel("Iteration")
plt.ylabel("Log-likelihood")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

print(len(theta_evolution))
fig, ax = plt.subplots()
im = ax.imshow(theta_evolution[0], cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
title = ax.set_title("Step 0")

def update(frame):
    im.set_data(theta_evolution[frame])
    title.set_text(f"Step {frame}")
    return im, title

anim = FuncAnimation(fig, update, frames=len(theta_evolution), interval=50)
anim.save("theta_evolution.gif", writer=PillowWriter(fps=200))
plt.close()